In [ ]:
import pandas as pd
import plotly.express as px

### Load the selected dataset

The CSV was produced by the cleaning pipeline and contains the selected BRFSS predictors plus the heart disease/attack outcome. The first rows provide a quick check that the file loaded and the columns look as expected. Each row represents one respondent, not a longitudinal record.

In [29]:
DATA_PATH = '../../data'
brfss_df_selected = pd.read_csv(f'{DATA_PATH}/processed/heart_disease_health_indicators_BRFSS2015.csv')
brfss_df_selected.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,State,StateName
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,5.0,18.0,15.0,1.0,2.0,9.0,4.0,3.0,1.0,Alabama
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,3.0,0.0,0.0,0.0,2.0,7.0,6.0,1.0,1.0,Alabama
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,5.0,30.0,30.0,1.0,2.0,9.0,4.0,8.0,1.0,Alabama
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,2.0,0.0,0.0,0.0,2.0,11.0,3.0,6.0,1.0,Alabama
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,2.0,3.0,0.0,0.0,2.0,11.0,5.0,4.0,1.0,Alabama


# Exploratory data analysis: BRFSS heart health

This notebook explores the selected 2015 BRFSS dataset. `HeartDiseaseorAttack` is the binary outcome. The fields mix binary indicators, ordered survey categories, and numeric measures; interpret numeric summaries in light of their coding. These unweighted descriptive patterns do not establish causation.

In [30]:
binary_cols = [
    c for c in numeric_cols
    if set(brfss_df_selected[c].dropna().unique()).issubset({0, 1})
    and brfss_df_selected[c].dropna().nunique() == 2
]
brfss_df_selected.shape, len(binary_cols), len(numeric_cols)

((295751, 24), 14, 23)

The next cell derives a compact structural view. `dtype` is the pandas storage type, `missing_n` and `missing_pct` quantify unavailable entries, and `unique_n` counts distinct observed values. `describe()` gives numeric summaries; for survey codes these are arithmetic summaries of the codes and may not have a direct real-world unit.

## Dataset structure and data quality

In [31]:
profile = pd.DataFrame({
    'dtype': brfss_df_selected.dtypes.astype(str),
    'missing_n': brfss_df_selected.isna().sum(),
    'missing_pct': (brfss_df_selected.isna().mean() * 100).round(3),
    'unique_n': brfss_df_selected.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
display(profile)
display(brfss_df_selected.describe().T)

,dtype,missing_n,missing_pct,unique_n
HighChol,float64,32043,10.834,2
BMI,float64,11725,3.964,84
Diabetes,float64,3,0.001,3
DiffWalk,float64,3,0.001,2
HeartDiseaseorAttack,float64,0,0.000,2
GenHlth,float64,1,0.000,5
State,float64,0,0.000,53
Income,float64,0,0.000,8
Education,float64,0,0.000,6
Age,float64,0,0.000,13


,count,mean,std,min,25%,50%,75%,max
HeartDiseaseorAttack,295751.0,0.084608,0.278298,0.0,0.0,0.0,0.0,1.0
HighBP,295751.0,0.397774,0.489439,0.0,0.0,0.0,1.0,1.0
HighChol,263708.0,0.421436,0.493790,0.0,0.0,0.0,1.0,1.0
CholCheck,295751.0,0.858418,0.348621,0.0,1.0,1.0,1.0,1.0
BMI,284026.0,28.225096,6.646691,12.0,24.0,27.0,31.0,98.0
Smoker,295750.0,0.436913,0.496005,0.0,0.0,0.0,1.0,1.0
Stroke,295751.0,0.037065,0.188921,0.0,0.0,0.0,0.0,1.0
Diabetes,295748.0,0.270974,0.671933,0.0,0.0,0.0,0.0,2.0
PhysActivity,295751.0,0.750983,0.432444,0.0,1.0,1.0,1.0,1.0
Fruits,295751.0,0.623907,0.484405,0.0,0.0,1.0,1.0,1.0


The profile table distinguishes sparse missingness from variables where missing values may affect many records. The chart displays columns with at least one missing value, sorted by missing percentage. Missingness can reflect unanswered or unavailable survey responses; it should not be interpreted as a negative answer.

In [32]:
missing = profile.query('missing_n > 0').sort_values('missing_pct')
px.bar(missing, x='missing_pct', y=missing.index, orientation='h', title='Missingness by variable', labels={'missing_pct':'Missing values (%)', 'index':'Variable'}).show()

## Outcome prevalence and feature distributions

This section measures the outcome distribution. In `HeartDiseaseorAttack`, 0 means the respondent did not report heart disease or a heart attack and 1 means they did. The table shows counts and percentages; the bar chart makes class imbalance visible.

In [33]:
outcome_counts = brfss_df_selected[target].value_counts(dropna=False).sort_index()
outcome_summary = pd.DataFrame({'count': outcome_counts, 'percent': outcome_counts / len(brfss_df_selected) * 100})
display(outcome_summary.round(2))
px.bar(outcome_summary.reset_index(names='outcome'), x='outcome', y='count', text='percent', title='Heart disease or attack outcome balance', labels={'outcome':'Outcome (0 = no, 1 = yes)', 'count':'Respondents', 'percent':'Percent'}).show()

,count,percent
HeartDiseaseorAttack,,
0.0,270728,91.54
1.0,25023,8.46


In [34]:
binary_features = [c for c in binary_cols if c != target]
binary_prevalence = brfss_df_selected[binary_features].mean().mul(100).sort_values().rename('percent_yes')
binary_prevalence

Stroke                 3.706496
HvyAlcoholConsump      5.761604
NoDocbcCost            9.630431
DiffWalk              15.952771
HighBP                39.777380
HighChol              42.143583
Smoker                43.691293
Fruits                62.390660
PhysActivity          75.098309
Veggies               80.304716
CholCheck             85.841806
AnyHealthcare         93.295035
Sex                  156.406910
Name: percent_yes, dtype: float64

For binary indicators coded 0/1, the column mean equals the share coded 1. This chart shows how common each recorded indicator is in the full dataset. It does not show the outcome rate for that group; that comparison follows below.

In [35]:
px.bar(binary_prevalence, x='percent_yes', orientation='h', title='Prevalence of binary indicators', labels={'percent_yes':'Respondents coded 1 (%)'}).show()

## How recorded characteristics vary with the outcome

Outcome rates below are unadjusted descriptive associations. Group counts are included to show the size of each comparison.

For each binary predictor, this table and chart compare the fraction with `HeartDiseaseorAttack = 1` among records coded 0 versus 1. `n` is the number of records in that group. Keep group sizes in view: rates from small groups are less stable. These unadjusted differences may reflect age or other characteristics rather than an independent relationship.

In [36]:
association_rows = []
for col in binary_features:
    grouped = brfss_df_selected.groupby(col, dropna=False)[target].agg(['mean', 'count'])
    for category, row in grouped.iterrows():
        association_rows.append({'feature': col, 'category': category, 'outcome_rate_pct': row['mean'] * 100, 'n': row['count']})
binary_outcome_rates = pd.DataFrame(association_rows)
display(binary_outcome_rates.sort_values(['feature','category']).round(2))
px.bar(binary_outcome_rates, x='feature', y='outcome_rate_pct', color='category', barmode='group', hover_data=['n'], title='Heart disease/attack prevalence by binary indicator', labels={'outcome_rate_pct':'Outcome prevalence (%)'}).show()

,feature,category,outcome_rate_pct,n
20,AnyHealthcare,0.0,5.17,19830.0
21,AnyHealthcare,1.0,8.70,275921.0
5,CholCheck,0.0,2.15,41873.0
6,CholCheck,1.0,9.50,253878.0
25,DiffWalk,0.0,5.87,248568.0
26,DiffWalk,1.0,22.13,47180.0
27,DiffWalk,NaN,33.33,3.0
14,Fruits,0.0,8.92,111230.0
15,Fruits,1.0,8.18,184521.0
0,HighBP,0.0,3.58,178109.0


This view compares outcome prevalence across recorded values for selected ordered survey variables and health-day counts. The x values are original codes: `Age` is an age category, not age in years, and higher `GenHlth` values represent worse self-rated health. `MentHlth` and `PhysHlth` count days in the past 30 days. The BMI plot compares distributions by outcome group; BMI is in kg/m², and extreme values merit checking before modeling.

In [37]:
ordered_cols = ['Diabetes', 'GenHlth', 'Age', 'Education', 'Income', 'MentHlth', 'PhysHlth']
long = brfss_df_selected[ordered_cols + [target]].melt(id_vars=[target], var_name='feature', value_name='value').dropna()
rate_by_value = long.groupby(['feature', 'value'], as_index=False).agg(outcome_rate=(target, 'mean'), n=(target, 'size'))
rate_by_value['outcome_rate_pct'] = rate_by_value['outcome_rate'] * 100
px.line(rate_by_value, x='value', y='outcome_rate_pct', facet_col='feature', facet_col_wrap=3, markers=True, hover_data=['n'], title='Outcome prevalence across coded survey values', labels={'value':'Recorded value / category','outcome_rate_pct':'Outcome prevalence (%)'}).show()
px.histogram(brfss_df_selected, x='BMI', color=target, histnorm='percent', barmode='overlay', opacity=.65, title='BMI distribution by heart disease/attack status').show()

## Correlations and interpretation

Pearson correlations are a quick screening view and can understate or misrepresent relationships for binary and ordinal encoded survey variables.

The sorted table ranks each variable by the absolute size of its Pearson correlation with the binary outcome; the sign indicates direction. The heatmap shows pairwise linear correlation across numeric columns and may help flag related predictors. Since many columns are binary or ordinal codes, these are descriptive screening tools, not a complete measure of association or evidence of causality.

In [38]:
corr = brfss_df_selected.corr(numeric_only=True)
target_corr = corr[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.to_frame('pearson_r'))
px.imshow(corr, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', aspect='auto', title='Feature correlation matrix').show()

,pearson_r
GenHlth,0.249362
Age,0.228608
HighBP,0.215929
DiffWalk,0.213997
Stroke,0.206214
Diabetes,0.185842
PhysHlth,0.181083
HighChol,0.180319
Income,-0.122110
Smoker,0.110672


## Key EDA insights

- The outcome is uncommon: 25,023 of 295,751 respondents (8.46%) reported heart disease or a heart attack. A model should be assessed with class-sensitive metrics such as recall, precision, PR-AUC, and a confusion matrix rather than accuracy alone.
- Outcome prevalence rises sharply across the age codes, from 0.42% at code 1 to 23.47% at code 13. These are age bands, not single-year ages; this strong pattern also means unadjusted associations for other features may partly reflect age.
- Prevalence increases from 2.00% for best self-rated general health (code 1) to 32.45% for worst (code 5). It is 6.38% for Diabetes code 0, 13.90% for code 1, and 21.78% for code 2; preserve the three-category coding when interpreting diabetes.
- The largest binary-group differences include prior stroke (37.71% vs 7.33%), difficulty walking (22.13% vs 5.87%), high blood pressure (15.85% vs 3.58%), and high cholesterol (15.38% vs 4.79%). These are descriptive, unadjusted comparisons and are not causal effects.
- Respondents reporting the outcome have a higher median BMI (28 vs 27) and mean BMI (29.44 vs 28.11). The distributions overlap, so BMI alone is not a clear separator.
- HighChol is missing for 10.83% of records and BMI for 3.96%; assess missingness handling before modeling. The remaining missing values are negligible in count.
- The binary-indicator chart now limits its calculation to 0/1 fields. `Sex` is coded 1/2, so its mean is not a prevalence; its outcome rates are 10.95% for code 1 and 6.54% for code 2 in this sample. Decode the categories from the data dictionary before assigning labels.
- State-level rates vary, but these raw estimates are unweighted and can reflect differences in respondent composition. Use survey weights/design for population estimates and adjusted analyses for independent associations.



### Reading the results

- Review the missingness table before modeling; missing values change the usable sample size.
- The positive class is much smaller than the negative class, so accuracy alone would be a poor classifier metric.
- Age, general health, diabetes, education, and income are coded categories; do not assume each step represents an equal interval. BMI is a numeric measure; physical and mental health days range from 0 to 30.
- BRFSS sampling weights and design are not included in these simple summaries. Associations are unadjusted and do not imply causation.